In [2]:
import pandas as pd
import numpy as np
import re
import category_encoders as ce

# ==============================================================================
# 1. LOAD DATA & FILTER LATEST SNAPSHOT (Strategi Baru)
# ==============================================================================
print("1. Loading Data...")
# Asumsi file awal excel, jika csv ganti read_csv
try:
    df = pd.read_excel('data/data_2_MCU.xlsx', engine='openpyxl')
except:
    # Fallback jika user menggunakan csv dari step sebelumnya
    try:
        df = pd.read_csv('data/data_2_MCU.csv')
    except:
        print("File tidak ditemukan, pastikan nama file benar.")
        raise

# Drop kolom dengan missing value ekstrem (>3000)
missing_value = df.isnull().sum()
cols_to_drop = missing_value[missing_value > 3000].index
df.drop(columns=cols_to_drop, inplace=True)

# --- CORE LOGIC: LATEST SNAPSHOT ---
print("   Applying 'Latest Snapshot' Strategy...")
if 'TANGGAL' in df.columns:
    df['TANGGAL'] = pd.to_datetime(df['TANGGAL'])
    # Urutkan berdasarkan BADGE dan TANGGAL (Terbaru di atas)
    df.sort_values(by=['BADGE', 'TANGGAL'], ascending=[True, False], inplace=True)
    # Ambil baris pertama untuk setiap BADGE (karena sudah diurutkan desc, ini adalah kunjungan terakhir)
    df = df.drop_duplicates(subset='BADGE', keep='first')
else:
    print("WARNING: Kolom 'TANGGAL' tidak ditemukan. Menggunakan baris terakhir per BADGE sebagai asumsi.")
    df = df.drop_duplicates(subset='BADGE', keep='last')

# Reset index agar rapi
df.reset_index(drop=True, inplace=True)

# ==============================================================================
# 2. TEXT CLEANING & STANDARDIZATION
# ==============================================================================
print("2. Cleaning Text Data...")
# Ambil kolom object/kategorikal
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].replace({
        'tidak diperiksa': 'tidak_diperiksa',
        'tidak periksa': 'tidak_diperiksa',
        'tidak di periksa': 'tidak_diperiksa',
        'tidak diperiksa ': 'tidak_diperiksa',
        'none': 'tidak_diperiksa',
        'nan': 'tidak_diperiksa',
        '-': 'tidak_diperiksa'
    })

# ==============================================================================
# 3. FEATURE ENGINEERING (Parsing Kompleks)
# ==============================================================================
print("3. Feature Engineering...")

# --- A. TENSI -> SISTOLIK & DIASTOLIK ---
def parse_tensi(x):
    if pd.isna(x) or x == 'tidak_diperiksa': return np.nan, np.nan
    s = str(x).replace(' ', '')
    if '/' not in s: return np.nan, np.nan
    try:
        a, b = s.split('/', 1)
        return float(a), float(b)
    except:
        return np.nan, np.nan

if 'TENSI' in df.columns:
    df[['SISTOLIK', 'DIASTOLIK']] = df['TENSI'].apply(lambda x: pd.Series(parse_tensi(x)))
    df.drop(columns=['TENSI'], inplace=True)

# --- B. KEHAMILAN -> TRIMESTER ---
def parse_trimester(x):
    s = str(x)
    if 'tidak_diperiksa' in s or '0' in s or '-' in s: return 0
    m = re.search(r'(\d+)\s*(minggu|mgg|bulan)', s)
    if not m: return 0
    num = int(m.group(1))
    weeks = num * 4.3 if 'bulan' in m.group(2) else num
    return 1 if weeks <= 13 else 2 if weeks <= 28 else 3

if 'KEHAMILAN' in df.columns:
    df['TRIMESTER'] = df['KEHAMILAN'].apply(parse_trimester)
    df.drop(columns=['KEHAMILAN'], inplace=True)

# --- C. URINE MICROSCOPIC ---
def parse_microscopic(val):
    s = str(val)
    if s in ['tidak_diperiksa', 'nan', '']: return np.nan
    # Cari angka/range (e.g., "0-1", "5-10")
    nums = re.findall(r'\d+\.?\d*', s)
    if nums:
        return np.mean([float(n) for n in nums])
    # Mapping kata
    mapping = {'nihil':0, 'negatif':0, 'positif':1, '+':3, '++':8, '+++':15, 'penuh':20}
    for k, v in mapping.items():
        if k in s: return v
    return np.nan

urin_cols = ['ERITROSIT_RBC', 'LEKOSIT_WBC', 'SEL_EPITEL']
for c in urin_cols:
    if c in df.columns:
        df[c] = df[c].apply(parse_microscopic)

# ==============================================================================
# 4. ENCODING (Binary & Ordinal)
# ==============================================================================
print("4. Encoding Variables...")

binary_map = {
    'OLAHRAGA': {'+': 1, '-': 0},
    'ALERGI': {'+': 1, '-': 0},
    'CONJUNGTIVA': {'normal': 0, 'anemis': 1},
    'HAEMORROID': {'normal': 0, 'ada': 1},
    'LEHER': {'normal': 0, 'pembesaran kgb': 1},
    'IRAMA_JANTUNG': {'normal': 0, 'ireguler': 1},
    'TREMOR': {'normal': 0, 'ada': 1},
    'PERUT': {'normal': 0, 'nyeri ketok va kiri': 1}, # Contoh spesifik dari data
    'LIMPA': {'normal': 0, 'teraba': 1},
    'HERNIA': {'normal': 0, 'ada': 1}
}

ordinal_map = {
    'UROBILINOGEN': {'negatif':0, 'normal':0},
    'BILIRUBIN': {'negatif':0},
    'PROTEIN_ALBUMIN': {'negatif':0, 'positif 1':1, 'positif':1, 'positif 2':2, '++':2},
    'REDUKSI': {'negatif':0, 'positif 1':1, 'positif':1},
    'GLUKOSA_URIN': {'negatif':0, 'positif 1':1, 'positif':1} # Sesuaikan nama kolom
}

# Terapkan mapping
for col, mp in {**binary_map, **ordinal_map}.items():
    if col in df.columns:
        # Map values, sisa yang tidak ada di map akan jadi NaN
        df[col] = df[col].map(mp)

# ==============================================================================
# 5. GROUPING & SIMPLIFICATION (Fitur Agregat Medis)
# ==============================================================================
print("5. Grouping Medical Features...")

# Fungsi bantu untuk grouping: Jika ada keyword abnormal di kolom-kolom target -> 1, else 0
def create_group_flag(df, cols, abnormal_keywords=['radang', 'sakit', 'polyp', 'devi', 'kotor', 'serumen', 'caries']):
    # Karena kita sudah lower(), kita cek string contains
    mask = pd.Series(0, index=df.index)
    existing_cols = [c for c in cols if c in df.columns]
    
    for col in existing_cols:
        # Jika kolom sudah numerik (hasil encoding), anggap > 0 adalah abnormal
        if pd.api.types.is_numeric_dtype(df[col]):
            mask = mask | (df[col] > 0)
        else:
            # Jika masih object, cek keyword 'normal' atau 'tidak_diperiksa' sebagai 0
            # Apapun yang BUKAN normal/tidak_diperiksa/nan dianggap abnormal
            is_normal = df[col].isin(['normal', 'tidak_diperiksa', 'nan'])
            mask = mask | (~is_normal)
    return mask.astype(int)

# Definisi Grup
df['ANY_EAR_ABNORMAL'] = create_group_flag(df, ['TELINGA', 'MEMBRAN_TYMPANI', 'SERUMEN_PLUG'])
df['ANY_NOSE_ABNORMAL'] = create_group_flag(df, ['HIDUNG', 'SEPTUM_DEVIASI', 'CONCHA', 'POLYP'])
df['ANY_THROAT_ABNORMAL'] = create_group_flag(df, ['KERONGKONGAN', 'TONSIL', 'FARING'])
df['ANY_MOUTH_ABNORMAL'] = create_group_flag(df, ['MULUT', 'GUSI'])

# ==============================================================================
# 6. FINAL COLUMN SELECTION & IMPUTATION
# ==============================================================================
print("6. Finalizing & Imputing...")

# Hapus kolom text asli yang sudah digrup/diparsing atau kolom tidak berguna untuk clustering
cols_to_drop_final = ['BAGIAN', 'TANGGAL', 'KULIT_RAMBUT', 'VISUS_MATA', 'AUDIOGRAM', 'ECG', 'TREADMILL_TEST', 'AUTOSPIROMETRI']
# Tambahkan kolom asli yang sudah di-group
cols_to_drop_final += ['TELINGA', 'MEMBRAN_TYMPANI', 'SERUMEN_PLUG', 'HIDUNG', 'SEPTUM_DEVIASI', 'CONCHA', 'POLYP', 'KERONGKONGAN', 'TONSIL', 'FARING', 'MULUT', 'GUSI']

df_final = df.drop(columns=[c for c in cols_to_drop_final if c in df.columns])

# Pastikan semua kolom numerik (kecuali BADGE)
cols_numeric = df_final.columns.drop('BADGE')

# Paksa ke numerik (errors='coerce' akan mengubah sisa string jadi NaN)
for col in cols_numeric:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

# Imputasi: Median untuk numerik, 0 untuk binary flag hasil grouping
for col in df_final.columns:
    if col == 'BADGE': continue
    if df_final[col].isnull().sum() > 0:
        median_val = df_final[col].median()
        df_final[col].fillna(median_val, inplace=True)

# Hitung BMI Akhir (Opsional jika TINGGI/BERAT ada)
if 'TINGGI' in df_final.columns and 'BERAT' in df_final.columns:
    # Asumsi satuan data mentah: Tinggi (mm e.g. 1660), Berat (hg e.g. 800)
    # Konversi ke meter dan kg
    t_m = df_final['TINGGI'] / 1000 # 1660 -> 1.66
    b_kg = df_final['BERAT'] / 10   # 800 -> 80
    df_final['BMI_CALC'] = b_kg / (t_m ** 2)
    df_final['BMI_CALC'] = df_final['BMI_CALC'].fillna(df_final['BMI_CALC'].median())

# Hapus kolom yang variance-nya 0 (semua nilai sama)
uniq_counts = df_final.nunique()
drop_const = uniq_counts[uniq_counts <= 1].index.tolist()
df_final.drop(columns=drop_const, inplace=True)

# ==============================================================================
# 7. SAVE
# ==============================================================================
output_filename = 'Cleaned_Pasien_MCU_LastVisit.csv'
print(f"7. Saving to {output_filename}...")
df_final.to_csv(output_filename, index=False)

print(f"✅ SUCCESS! Data cleaned saved. Shape: {df_final.shape}")
print(f"   Columns: {list(df_final.columns[:10])}...")

1. Loading Data...
   Applying 'Latest Snapshot' Strategy...
2. Cleaning Text Data...
3. Feature Engineering...
4. Encoding Variables...
5. Grouping Medical Features...
6. Finalizing & Imputing...
7. Saving to Cleaned_Pasien_MCU_LastVisit.csv...
✅ SUCCESS! Data cleaned saved. Shape: (1888, 51)
   Columns: ['BADGE', 'TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'CONJUNGTIVA', 'HAEMORROID', 'LEHER', 'IRAMA_JANTUNG']...


C:\Users\Loq Gaming\AppData\Local\Temp\ipykernel_45220\1592287673.py:197: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_final[col].fillna(median_val, inplace=True)
